# Gemma 3 轻量大模型推理优化与质量评测

运行前请在 Colab 中选择 GPU，并在 Hugging Face 接受 `google/gemma-3-1b-it` 的许可。随后在 Colab Secrets 中创建名为 `HF_TOKEN` 的密钥。不要把 Token 写进代码或提交到 GitHub。

In [ ]:
REPO_URL = 'https://github.com/zmh2245749337/gemma-inference-eval.git'
!git clone {REPO_URL}
%cd gemma-inference-eval
!pip -q install -r requirements-colab.txt

In [ ]:
import os
from google.colab import userdata
token = userdata.get('HF_TOKEN')
if not token:
    raise ValueError('没有读取到 HF_TOKEN，请在 Colab Secrets 中添加。')
os.environ['HF_TOKEN'] = token
!nvidia-smi

## 1. 手写解码与 Hugging Face 对齐

先使用贪心解码，检查两种实现是否逐 token 一致。

In [ ]:
!python scripts/run_manual_decode.py --prompt '请用三句话解释什么是KV Cache。' --max-new-tokens 48 --greedy --check-hf-parity --output reports/manual_decode.json

## 2. KV Cache 基准

脚本会自动比较开启和关闭 Cache。先用较短输出确认流程，再增加长度。

In [ ]:
!python scripts/run_benchmark.py --precision fp16 --max-new-tokens 32 --warmups 1 --repeats 3 --output reports/benchmark.csv

## 3. 四位量化基准

若前一个模型仍占显存，建议重启运行时后直接从安装和登录单元格继续，再执行下面命令。

In [ ]:
!python scripts/run_benchmark.py --precision 4bit --max-new-tokens 32 --warmups 1 --repeats 3 --output reports/benchmark.csv

## 4. 固定题集质量检查

不同精度应使用不同输出文件，避免覆盖证据。

In [ ]:
!python scripts/run_quality_eval.py --precision 4bit --dataset configs/prompts_zh.jsonl --output reports/quality_4bit.jsonl

In [ ]:
import pandas as pd
benchmark = pd.read_csv('reports/benchmark.csv')
display(benchmark[['precision', 'use_kv_cache', 'ttft_ms_median', 'decode_tps_median', 'total_tps_median', 'peak_memory_mb']])